# House Price Prediction: End-to-End Regression Analysis

An end-to-end machine learning project for predicting residential sale prices using EDA, preprocessing pipelines, regression models, hyperparameter tuning, cross-validation, residual analysis, and feature importance.


## 1. Imports and Data Loading


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('train.csv')
df.head()


## 2. Data Understanding


In [ ]:
print('Shape:', df.shape)
df.info()


In [ ]:
df.isnull().sum().sort_values(ascending=False)[lambda s: s > 0]


## 3. Target Analysis

`SalePrice` is right-skewed, so we use `log1p` for the modeling target.


In [ ]:
print(df['SalePrice'].describe())
print('Median:', df['SalePrice'].median())
print('Original skewness:', df['SalePrice'].skew())
df['SalePrice_log'] = np.log1p(df['SalePrice'])
print('Log skewness:', df['SalePrice_log'].skew())


In [ ]:
plt.figure(figsize=(8,5)); plt.hist(df['SalePrice'], bins=30); plt.xlabel('SalePrice'); plt.ylabel('Frequency'); plt.title('SalePrice Distribution'); plt.show()
plt.figure(figsize=(8,5)); plt.hist(df['SalePrice_log'], bins=30); plt.xlabel('Log(SalePrice)'); plt.ylabel('Frequency'); plt.title('Log-Transformed SalePrice'); plt.show()


## 4. Exploratory Data Analysis

### Overall Quality


In [ ]:
print(df.groupby('OverallQual')['SalePrice'].median())
print(df.groupby('OverallQual')['SalePrice'].mean())


In [ ]:
plt.figure(figsize=(8,5)); plt.scatter(df['OverallQual'], df['SalePrice'], alpha=0.5); plt.xlabel('OverallQual'); plt.ylabel('SalePrice'); plt.title('Overall Quality vs Sale Price'); plt.show()


### Living Area and Neighborhood


In [ ]:
print(df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False))
plt.figure(figsize=(8,5)); plt.scatter(df['GrLivArea'], df['SalePrice'], alpha=0.5); plt.xlabel('GrLivArea'); plt.ylabel('SalePrice'); plt.title('Living Area vs Sale Price'); plt.show()


In [ ]:
df.select_dtypes(include=['int64','float64']).corr()['SalePrice'].sort_values(ascending=False)


## 5. Features, Target, and Feature Types

`Id` is removed because it is an identifier. `MSSubClass` is treated as categorical because its integer codes represent property classes rather than a continuous quantity.


In [ ]:
X = df.drop(columns=['SalePrice','SalePrice_log','Id'])
y = df['SalePrice_log']
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
categorical_features.append('MSSubClass')
numerical_features = X.select_dtypes(include=['int64','float64']).columns.tolist()
numerical_features.remove('MSSubClass')
print('X:', X.shape, 'y:', y.shape)
print('Categorical:', len(categorical_features))
print('Numerical:', len(numerical_features))


## 6. Train/Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)


## 7. Preprocessing Pipeline

Numerical features: median imputation + scaling. Categorical features: most-frequent imputation + one-hot encoding. All preprocessing is fitted inside each model pipeline to avoid leakage.


In [ ]:
numeric_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('num', numeric_transformer, numerical_features), ('cat', categorical_transformer, categorical_features)])
preprocessor


## 8. Linear Regression Baseline


In [ ]:
linear_model = Pipeline([('preprocessor', preprocessor), ('model', LinearRegression())])
linear_model.fit(X_train, y_train)
y_pred_log = linear_model.predict(X_test)
mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
r2_log = r2_score(y_test, y_pred_log)
y_test_actual = np.expm1(y_test); y_pred = np.expm1(y_pred_log)
mae = mean_absolute_error(y_test_actual, y_pred); rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))
print('MAE (log):', mae_log); print('RMSE (log):', rmse_log); print('R² (log):', r2_log); print('MAE ($):', mae); print('RMSE ($):', rmse)


## 9. Ridge Regression


In [ ]:
ridge_model = Pipeline([('preprocessor', preprocessor), ('model', Ridge())])
ridge_model.fit(X_train, y_train); y_pred_ridge_log = ridge_model.predict(X_test); y_pred_ridge = np.expm1(y_pred_ridge_log)
mae_ridge_log = mean_absolute_error(y_test,y_pred_ridge_log); rmse_ridge_log=np.sqrt(mean_squared_error(y_test,y_pred_ridge_log)); r2_ridge_log=r2_score(y_test,y_pred_ridge_log)
mae_ridge=mean_absolute_error(y_test_actual,y_pred_ridge); rmse_ridge=np.sqrt(mean_squared_error(y_test_actual,y_pred_ridge))
print('Ridge MAE (log):',mae_ridge_log); print('Ridge RMSE (log):',rmse_ridge_log); print('Ridge R² (log):',r2_ridge_log); print('Ridge MAE ($):',mae_ridge); print('Ridge RMSE ($):',rmse_ridge)


## 10. Decision Tree Regression


In [ ]:
dt_model = Pipeline([('preprocessor', preprocessor), ('model', DecisionTreeRegressor(random_state=42))])
dt_model.fit(X_train,y_train); y_pred_dt_log=dt_model.predict(X_test); y_pred_dt=np.expm1(y_pred_dt_log)
mae_dt_log=mean_absolute_error(y_test,y_pred_dt_log); rmse_dt_log=np.sqrt(mean_squared_error(y_test,y_pred_dt_log)); r2_dt_log=r2_score(y_test,y_pred_dt_log)
mae_dt=mean_absolute_error(y_test_actual,y_pred_dt); rmse_dt=np.sqrt(mean_squared_error(y_test_actual,y_pred_dt))
print('Decision Tree MAE (log):',mae_dt_log); print('Decision Tree RMSE (log):',rmse_dt_log); print('Decision Tree R² (log):',r2_dt_log); print('Decision Tree MAE ($):',mae_dt); print('Decision Tree RMSE ($):',rmse_dt)


## 11. Random Forest Baseline


In [ ]:
rf_model = Pipeline([('preprocessor',preprocessor),('model',RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1))])
rf_model.fit(X_train,y_train); y_pred_rf_log=rf_model.predict(X_test); y_pred_rf=np.expm1(y_pred_rf_log)
mae_rf_log=mean_absolute_error(y_test,y_pred_rf_log); rmse_rf_log=np.sqrt(mean_squared_error(y_test,y_pred_rf_log)); r2_rf_log=r2_score(y_test,y_pred_rf_log)
mae_rf=mean_absolute_error(y_test_actual,y_pred_rf); rmse_rf=np.sqrt(mean_squared_error(y_test_actual,y_pred_rf))
print('Random Forest MAE (log):',mae_rf_log); print('Random Forest RMSE (log):',rmse_rf_log); print('Random Forest R² (log):',r2_rf_log); print('Random Forest MAE ($):',mae_rf); print('Random Forest RMSE ($):',rmse_rf)


## 12. Model Comparison


In [ ]:
results = pd.DataFrame({'Model':['Linear Regression','Ridge','Decision Tree','Random Forest'],'R2_Log':[r2_log,r2_ridge_log,r2_dt_log,r2_rf_log],'MAE':[mae,mae_ridge,mae_dt,mae_rf],'RMSE':[rmse,rmse_ridge,rmse_dt,rmse_rf]})
results.sort_values('MAE')


## 13. Random Forest Hyperparameter Tuning


In [ ]:
rf_pipeline = Pipeline([('preprocessor',preprocessor),('model',RandomForestRegressor(random_state=42,n_jobs=-1))])
param_grid_rf={'model__n_estimators':[200,300],'model__max_depth':[None,10,20],'model__min_samples_split':[2,5],'model__min_samples_leaf':[1,2],'model__max_features':['sqrt',0.7]}
grid_rf=GridSearchCV(rf_pipeline,param_grid=param_grid_rf,cv=5,scoring='neg_root_mean_squared_error',n_jobs=-1,verbose=1)
grid_rf.fit(X_train,y_train)
print('Best parameters:',grid_rf.best_params_); print('Best CV RMSE:',-grid_rf.best_score_)


In [ ]:
y_pred_rf_tuned_log=grid_rf.predict(X_test); y_pred_rf_tuned=np.expm1(y_pred_rf_tuned_log)
mae_rf_tuned_log=mean_absolute_error(y_test,y_pred_rf_tuned_log); rmse_rf_tuned_log=np.sqrt(mean_squared_error(y_test,y_pred_rf_tuned_log)); r2_rf_tuned_log=r2_score(y_test,y_pred_rf_tuned_log)
mae_rf_tuned=mean_absolute_error(y_test_actual,y_pred_rf_tuned); rmse_rf_tuned=np.sqrt(mean_squared_error(y_test_actual,y_pred_rf_tuned))
print('Tuned RF MAE (log):',mae_rf_tuned_log); print('Tuned RF RMSE (log):',rmse_rf_tuned_log); print('Tuned RF R² (log):',r2_rf_tuned_log); print('Tuned RF MAE ($):',mae_rf_tuned); print('Tuned RF RMSE ($):',rmse_rf_tuned)


## 14. Cross-Validation


In [ ]:
cv_scores_lr=cross_val_score(linear_model,X_train,y_train,cv=5,scoring='neg_root_mean_squared_error',n_jobs=-1)
cv_scores_rf=cross_val_score(grid_rf.best_estimator_,X_train,y_train,cv=5,scoring='neg_root_mean_squared_error',n_jobs=-1)
print('Linear Regression fold RMSE:',-cv_scores_lr); print('Mean:',-cv_scores_lr.mean()); print('Std:',cv_scores_lr.std())
print('\nTuned Random Forest fold RMSE:',-cv_scores_rf); print('Mean:',-cv_scores_rf.mean()); print('Std:',cv_scores_rf.std())


## 15. Actual vs Predicted and Residual Analysis


In [ ]:
plt.figure(figsize=(8,6)); plt.scatter(y_test_actual,y_pred_rf_tuned,alpha=0.6); plt.plot([y_test_actual.min(),y_test_actual.max()],[y_test_actual.min(),y_test_actual.max()],linestyle='--'); plt.xlabel('Actual Sale Price'); plt.ylabel('Predicted Sale Price'); plt.title('Actual vs Predicted — Tuned Random Forest'); plt.show()
residuals=y_test_actual-y_pred_rf_tuned
print('Mean residual:',residuals.mean()); print('Median residual:',residuals.median()); print('Minimum residual:',residuals.min()); print('Maximum residual:',residuals.max())
plt.figure(figsize=(8,6)); plt.scatter(y_pred_rf_tuned,residuals,alpha=0.6); plt.axhline(0,linestyle='--'); plt.xlabel('Predicted Sale Price'); plt.ylabel('Residual'); plt.title('Residuals vs Predicted Sale Price'); plt.show()


In [ ]:
error_analysis=pd.DataFrame({'ActualPrice':y_test_actual,'PredictedPrice':y_pred_rf_tuned,'Residual':residuals},index=y_test.index)
error_analysis['AbsoluteError']=error_analysis['Residual'].abs()
error_analysis.sort_values('AbsoluteError',ascending=False).head(10)


In [ ]:
worst_indices=error_analysis.sort_values('AbsoluteError',ascending=False).head(10).index
df.loc[worst_indices,['Id','SalePrice','OverallQual','GrLivArea','Neighborhood','YearBuilt','YearRemodAdd','GarageCars','GarageArea','TotalBsmtSF','1stFlrSF']]


## 16. Feature Importance

Feature importance is predictive importance within the Random Forest, not causal impact.


In [ ]:
preprocessor_fitted=grid_rf.best_estimator_.named_steps['preprocessor']
rf_fitted=grid_rf.best_estimator_.named_steps['model']
feature_names=preprocessor_fitted.get_feature_names_out()
importances=rf_fitted.feature_importances_
feature_importance=pd.DataFrame({'Feature':feature_names,'Importance':importances}).sort_values('Importance',ascending=False)
feature_importance.head(20)


In [ ]:
top_features=feature_importance.head(15).sort_values('Importance')
plt.figure(figsize=(10,7)); plt.barh(top_features['Feature'],top_features['Importance']); plt.xlabel('Feature Importance'); plt.ylabel('Feature'); plt.title('Top 15 Feature Importances — Tuned Random Forest'); plt.tight_layout(); plt.show()


## 17. Domain-Aware Missing-Value Experiment

Structural missing values were also tested as an explicit `None` category. This produced slightly worse held-out test performance than the original most-frequent strategy, so the original preprocessing was retained for the main model comparison.


In [ ]:
absence_features=['Alley','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','FireplaceQu','GarageType','GarageFinish','GarageQual','GarageCond','PoolQC','Fence','MiscFeature','MasVnrType']
cat_absence=[c for c in categorical_features if c in absence_features]
cat_other=[c for c in categorical_features if c not in absence_features]
num_pipe=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
absence_pipe=Pipeline([('imputer',SimpleImputer(strategy='constant',fill_value='None')),('encoder',OneHotEncoder(handle_unknown='ignore'))])
other_pipe=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('encoder',OneHotEncoder(handle_unknown='ignore'))])
preprocessor_improved=ColumnTransformer([('num',num_pipe,numerical_features),('cat_absence',absence_pipe,cat_absence),('cat_other',other_pipe,cat_other)])
linear_model_improved=Pipeline([('preprocessor',preprocessor_improved),('model',LinearRegression())])
linear_model_improved.fit(X_train,y_train); y_pred_improved_log=linear_model_improved.predict(X_test); y_pred_improved=np.expm1(y_pred_improved_log)
print('Improved Linear Regression MAE (log):',mean_absolute_error(y_test,y_pred_improved_log)); print('Improved Linear Regression RMSE (log):',np.sqrt(mean_squared_error(y_test,y_pred_improved_log))); print('Improved Linear Regression R² (log):',r2_score(y_test,y_pred_improved_log)); print('Improved Linear Regression MAE ($):',mean_absolute_error(y_test_actual,y_pred_improved)); print('Improved Linear Regression RMSE ($):',np.sqrt(mean_squared_error(y_test_actual,y_pred_improved)))


## 18. Final Results and Conclusion

### Held-out test set

Linear Regression achieved the strongest held-out test performance: R² = 0.9088, MAE ≈ $15,056, and RMSE ≈ $23,104.

### Cross-validation

Tuned Random Forest achieved stronger average 5-fold CV RMSE (0.1417 vs 0.1709 for Linear Regression) and lower fold-to-fold variability (0.0201 vs 0.0271).

### Key findings

- `OverallQual` was the strongest Random Forest feature at approximately 40.8% importance.
- `GrLivArea` was second at approximately 17.8%.
- Other influential variables included `YearBuilt`, `GarageCars`, `TotalBsmtSF`, `GarageArea`, and `1stFlrSF`.
- Some of the largest errors occurred among high-value properties, especially several premium properties in `NoRidge`, `NridgHt`, and `StoneBr`.

### Limitations

- The dataset contains 1,460 observations.
- Test performance depends on the selected train/test split.
- Some high-value properties are difficult to predict accurately.
- Feature importance is predictive, not causal.
- Historical data may not represent current housing markets.
